# 02 · Train a small model (Colab-friendly)
Trains the attention-based Seq2Seq model on the synthetic corpus. On Colab: Runtime > Change runtime type > GPU is optional (this is small enough for CPU) but will speed things up.

This mirrors what `python scripts/train.py --config experiments/configs/attn_bahdanau.yaml` does from the command line -- use whichever you prefer.

In [ ]:
# If running on Colab, uncomment to clone/copy the repo and install deps:
# !pip install -q torch --index-url https://download.pytorch.org/whl/cpu
# !pip install -q -r requirements.txt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))

In [ ]:
from src.config import load_config
from src.utils.seed import set_seed
from src.utils.io import read_jsonl
from src.datasets.pipeline import build_dataset
from src.datasets.torch_data import make_loader, mix_sources
from src.tokenization.factory import build_tokenizer
from src.models.factory import build_model, model_kwargs
from src.training.loop import train_model

cfg = load_config(base_path='../config.yaml')
set_seed(cfg['seed'])

In [ ]:
# Build the dataset if you haven't already (safe to re-run; deterministic).
stats = build_dataset(cfg, out_dir='../data/processed')
stats['n_pairs']

In [ ]:
train_rows = read_jsonl('../data/processed/train.jsonl')
valid_rows = read_jsonl('../data/processed/valid.jsonl')
src_tok, tgt_tok = build_tokenizer(cfg, [r['src'] for r in train_rows], [r['tgt'] for r in train_rows])
len(train_rows), len(valid_rows), src_tok.vocab_size, tgt_tok.vocab_size

In [ ]:
loader = make_loader(train_rows, src_tok, tgt_tok, cfg['train']['batch_size'], shuffle=True,
                     max_src_len=cfg['data']['max_src_len'], max_tgt_len=cfg['data']['max_tgt_len'], seed=cfg['seed'])
mk = model_kwargs(cfg['model'])
model = build_model(mk, src_tok, tgt_tok)
model._mk = mk
sum(p.numel() for p in model.parameters())

In [ ]:
# Shrink epochs here for a quick notebook run; raise for a real experiment.
cfg['train']['epochs'] = 15
cfg['train']['run_name'] = 'notebook_run'
result = train_model(model, src_tok, tgt_tok, loader, valid_rows, cfg['train'], device='cpu',
                     run_dir='../experiments/checkpoints/notebook_run')
result['best_epoch'], result['best_score']

## Try it on the gold demo sentences

In [ ]:
from src.inference.generate import translate_texts
from src.datasets.pipeline import load_split

gold = load_split('../data/processed', 'gold_demo')
hyps = translate_texts(model, src_tok, tgt_tok, [r['src'] for r in gold], device='cpu', beam_size=4)
for r, h in zip(gold, hyps):
    print(f"{r['src']!r:35s} -> {h[0]['text']!r:35s} (ref: {r['tgt']!r})")